In [ ]:
#READ EXCEL
import pandas as pd
import numpy as np

df = pd.read_excel("Cleaned_FIFA_Data.xlsx")

print(df.head())


In [ ]:
# GET UNIQUE CLUBS
# #clubs = df["Clubs"]
#print(clubs.head())

unique_clubs = df["Club"].unique()

print(np.sort(unique_clubs))


In [ ]:
# ENTER UNIQUE CLUBS
clubs_df = pd.DataFrame(np.sort(unique_clubs), columns= ["Club"])
clubs_df.dropna()
clubs_df.drop_duplicates()

print(clubs_df) 

In [ ]:
# FIX CLUB NAMES
clubs_df["Fixed_Name"] = clubs_df["Fixed_Name"].str.replace( "CD ", "Club Deportivo ")

clubs_df["Search_Name"] = (clubs_df["Fixed_Name"].str.replace(' ', '_'))

clubs_df["Crest_URL"] = ""

print(clubs_df)

In [ ]:
# TEST GET URL
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Granada_CF"

my_info_header = {
    "User-Agent": "MyWikipediaBot/1.0 (https://github.com/Vicky-Kounadi)"
}

response = requests.get(url, headers=my_info_header)
#print(response.status_code)
#print(response.text)

soup = BeautifulSoup(response.text, "html.parser")

#print ("*****")
#print(soup.title)

infobox = soup.find("table", class_="infobox")

img = infobox.find("img")

crest_url = img["src"]

if crest_url.startswith("//"):
    crest_url = "https:" + crest_url

print(crest_url)


In [ ]:
#TEST DOWNLOAD IMAGE
my_info_header = {
    "User-Agent": "MyWikipediaBot/1.0 (https://github.com/Vicky-Kounadi)"
}

img_response = requests.get(crest_url, headers=my_info_header)

print(img_response.status_code)
print(img_response.headers["Content-Type"])

with open("assets/crest_images/_crest.png", "wb") as f:
    f.write(img_response.content)


In [47]:
# FUNCTIONS GET URL N DOWNLOAD IMAGE
import requests
from bs4 import BeautifulSoup

my_info_header = {
    "User-Agent": "MyWikipediaBot/1.0 (https://github.com/Vicky-Kounadi)"
}

def get_crest_url(club):
        # CONNECT
        url = f"https://en.wikipedia.org/wiki/{club}"

        response = requests.get(url, headers=my_info_header)

        soup = BeautifulSoup(response.text, "html.parser")

        infobox = soup.find("table", class_="infobox")
        if infobox is None:
            print(f"No infobox: {club}")
            return None

        # GET IMAGE BLOCK IN INFOBOX
        img = infobox.find("img")
        if img is None:
            print(f"No image: {club}")
            return None

        # FIND SOURCE IMAGE, NOT HREF TO WEB
        crest_url = img["src"]
        if not crest_url:
            return None
        
        if crest_url.startswith("//"):
            crest_url = "https:" + crest_url

        print(crest_url)
        return crest_url

def download_crest_image(url, club):
        # INVALID LINK
        if pd.isna(url) or not url:
            url= "https://upload.wikimedia.org/wikipedia/commons/thumb/2/21/Solid_black.svg/500px-Solid_black.svg.png"
       
        # DOWNLOAD IMAGES            
        img_response = requests.get(url, headers=my_info_header)

        print(img_response.status_code)
        print(img_response.headers["Content-Type"])

        with open(f"assets/crest_images/{club}_crest.png", "wb") as f:
            f.write(img_response.content)



In [ ]:
# USE GET URL
clubs_df["Crest_URL"] = clubs_df["Search_Name"].apply(get_crest_url)

In [ ]:
# FUNCTION TO GET CLOSE NAME
import requests

my_info_header = {
    "User-Agent": "MyWikipediaBot/1.0 (https://github.com/Vicky-Kounadi)"
}

def wikipedia_search(club):
    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": club + " football club",
        "format": "json",
        "srlimit": 5
    }

    r = requests.get(url, params=params, headers=my_info_header).json()

    results = r["query"]["search"]

    if results:
        return results[0]["title"]

    return None


In [ ]:
#RESULTS OF WIKIPEDIA SEARCH
import time

results = []

for club in clubs_df["Club"]:
    result = wikipedia_search(club)
    print(club, "→", result)
    results.append(result)
    time.sleep(0.5)

clubs_df["Fixed_Name"] = results

In [ ]:
# EXPORT DATA URL TO CSV

clubs_df["Search_Name"] = (clubs_df["Fixed_Name"].str.replace(' ', '_'))
print(clubs_df)


#clubs_df.to_csv("club_crest_urls.csv", index=True)

In [ ]:
#GET URLS
clubs_df["Crest_URL"] = clubs_df["Search_Name"].apply(get_crest_url)

In [ ]:
#MAKE FINAL CSV
print(clubs_df)

clubs_df.to_csv("club_crest_urls.csv", index=True)

In [ ]:
# LOAD FIXED CSV N DOWNLOAD IMAGES
import pandas as pd

clubs_df.apply(
    lambda rec: download_crest_image(rec["Crest_URL"], rec["Search_Name"]),
    axis=1
)

In [ ]:
# LOAD FIXED CSV N DOWNLOAD IMAGES

clubs_fixed = pd.read_csv("club_crest_urls.csv")

clubs_fixed.apply(
    lambda rec: download_crest_image( rec["Crest_URL"], rec["Search_Name"]),
    axis=1
)